# UCS420 Cognitive Computing
## Assignment 10: Heart Health Information Assistant

Name: Ansh Sharma  
Roll number: 1024170344

The roll number is even, so this notebook builds a Heart Health Information Assistant. It classifies queries, retrieves general information, and escalates emergency language.

## Labelled corpus

In [ ]:
corpus = {
    'symptoms': [
        'What are common signs of heart disease?',
        'Why does my heart sometimes beat irregularly?',
        'Can a heart problem cause dizziness?',
        'What does a fast heartbeat feel like?',
        'How can I check if my pulse is irregular?',
        'Can high blood pressure have symptoms?'
    ],
    'risk_factors': [
        'Does family history increase heart disease risk?',
        'How does smoking affect the heart?',
        'Can high cholesterol raise cardiac risk?',
        'Does diabetes affect heart health?',
        'Why is high blood pressure a risk factor?',
        'Can stress affect cardiovascular health?'
    ],
    'prevention': [
        'How can I lower my risk of heart disease?',
        'What habits help protect the heart?',
        'How often should I get a heart health checkup?',
        'Can quitting smoking improve heart health?',
        'What can I do to prevent cardiovascular disease?',
        'How can regular checkups help my heart?'
    ],
    'nutrition': [
        'What foods are good for heart health?',
        'Should I reduce salt for my heart?',
        'What is a heart healthy diet?',
        'Are fruits and vegetables helpful for the heart?',
        'How can I eat less saturated fat?',
        'What drinks should I limit for heart health?'
    ],
    'exercise': [
        'What exercise supports a healthy heart?',
        'How much physical activity is recommended?',
        'Can walking help cardiovascular fitness?',
        'How should I start exercising safely?',
        'Does regular activity benefit the heart?',
        'What should I ask before starting a new workout?'
    ],
    'tests': [
        'What happens during an ECG test?',
        'What is an echocardiogram used for?',
        'Why would a doctor check cholesterol?',
        'What does a blood pressure test measure?',
        'How do doctors check heart rhythm?',
        'What should I ask about my heart test results?'
    ],
    'medications': [
        'Why might a doctor prescribe heart medicine?',
        'Should I stop taking my prescribed medication?',
        'What should I ask about medicine side effects?',
        'Can I change my heart medicine dose myself?',
        'How can I remember to take prescribed medicine?',
        'Who can explain how my medication works?'
    ],
    'emergency': [
        'What should I do for sudden severe chest pain?',
        'Someone is having trouble breathing and chest pressure.',
        'A person suddenly collapsed and is unresponsive.',
        'What do I do if someone may be having a heart attack?',
        'I have crushing chest pain right now.',
        'Someone fainted and is not breathing normally.'
    ]
}

queries = [query for examples in corpus.values() for query in examples]
labels = [intent for intent, examples in corpus.items() for query in examples]
print('Number of queries:', len(queries))
print('Queries per intent:')
print({intent: len(examples) for intent, examples in corpus.items()})

## Taxonomy and ontology

In [ ]:
taxonomy = {
    'Heart Information': ['symptoms', 'risk_factors'],
    'Prevention': ['prevention', 'nutrition'],
    'Healthy Living': ['exercise', 'tests'],
    'Care and Urgent Help': ['medications', 'emergency']
}

ontology = {
    'Person': ['has risk factor', 'experiences symptom', 'asks about test'],
    'Heart': ['has rhythm', 'is affected by disease'],
    'Lifestyle': ['includes nutrition', 'includes exercise'],
    'Medication': ['is prescribed by clinician'],
    'Test': ['measures heart information'],
    'Clinician': ['orders test', 'prescribes medication'],
    'Emergency': ['requires urgent help'],
    'RiskFactor': ['can increase risk of heart disease']
}

relationships = [
    ('Person', 'experiences', 'Symptom'),
    ('Person', 'has', 'RiskFactor'),
    ('RiskFactor', 'may increase risk of', 'Heart Disease'),
    ('Lifestyle', 'includes', 'Nutrition'),
    ('Lifestyle', 'includes', 'Exercise'),
    ('Clinician', 'may order', 'Test'),
    ('Test', 'measures', 'Heart information'),
    ('Clinician', 'may prescribe', 'Medication'),
    ('Medication', 'may support care for', 'Heart condition'),
    ('Emergency', 'requires contacting', 'Local emergency services')
]

print('Taxonomy:', taxonomy)
pd.DataFrame(relationships, columns=['Subject', 'Relationship', 'Object'])

## NLP preprocessing and intent model

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
tokenizer = RegexpTokenizer(r'[A-Za-z]+')

def preprocess(text):
    tokens = tokenizer.tokenize(text.lower())
    return ' '.join(lemmatizer.lemmatize(token) for token in tokens if token not in stop_words)

processed_queries = [preprocess(query) for query in queries]
print(queries[0])
print(processed_queries[0])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    processed_queries, labels, test_size=0.25, random_state=42, stratify=labels
)

vectorizer = TfidfVectorizer(tokenizer=str.split, preprocessor=None, token_pattern=None)
x_train_vectors = vectorizer.fit_transform(x_train)
x_test_vectors = vectorizer.transform(x_test)
classifier = LogisticRegression(max_iter=1000)
classifier.fit(x_train_vectors, y_train)
predictions = classifier.predict(x_test_vectors)

print('Accuracy:', round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions, labels=list(corpus.keys())))
print('Intent order:', list(corpus.keys()))

## Knowledge base and assistant

In [ ]:
knowledge_base = {
    'symptoms': 'Heart symptoms can vary. A clinician can assess symptoms and decide what checks may be needed.',
    'risk_factors': 'Risk factors can include family history, smoking, high blood pressure, high cholesterol, diabetes, and other health conditions. A clinician can explain personal risk.',
    'prevention': 'Regular checkups and healthy habits may support heart health. A clinician can help choose steps that fit your health needs.',
    'nutrition': 'A balanced eating pattern with vegetables, fruits, whole grains, and less excess salt may support heart health. Ask a clinician about your personal diet.',
    'exercise': 'Regular physical activity can support cardiovascular health. Ask a clinician before changing activity if you have symptoms or a health condition.',
    'tests': 'Common heart checks may include blood pressure measurement, blood tests, an ECG, or an echocardiogram. A clinician can explain which test is appropriate.',
    'medications': 'Take medicines as prescribed and ask a clinician or pharmacist about side effects or changes. Do not change a dose on your own.',
    'emergency': 'If someone may be having a heart emergency, contact local emergency services now.'
}

emergency_phrases = [
    'crushing chest pain', 'severe chest pain', 'cannot breathe',
    'trouble breathing', 'not breathing normally', 'collapsed and unresponsive',
    'heart attack right now'
]
domain_words = {'heart', 'cardiac', 'cardiovascular', 'blood pressure', 'pulse', 'cholesterol', 'ecg'}
confidence_threshold = 0.30

def assistant(query):
    normalized_query = query.lower()
    if any(phrase in normalized_query for phrase in emergency_phrases):
        return 'Emergency rule: contact local emergency services now.'
    if not any(term in normalized_query for term in domain_words):
        return 'Unsupported query: I can help with general heart health information.'
    processed = preprocess(query)
    vector = vectorizer.transform([processed])
    probabilities = classifier.predict_proba(vector)[0]
    best_index = probabilities.argmax()
    confidence = probabilities[best_index]
    intent = classifier.classes_[best_index]
    if confidence < confidence_threshold:
        return 'Low confidence: please rephrase your heart health question or ask a clinician.'
    if intent == 'emergency':
        return knowledge_base[intent]
    return knowledge_base[intent]

print(assistant('What foods are good for heart health?'))

## Test queries

In [ ]:
test_queries = [
    'What are common signs of heart disease?',
    'Does family history affect cardiac risk?',
    'How can I lower my risk of heart disease?',
    'Should I eat less salt for my heart?',
    'Can walking improve cardiovascular fitness?',
    'What happens during an ECG?',
    'Should I stop taking prescribed heart medicine?',
    'I have crushing chest pain right now.',
    'How can I protect my cardiovascular system?',
    'How do I repair a bicycle chain?',
    'Tell me about my heart thing.'
]

for query in test_queries:
    print('You:', query)
    print('Assistant:', assistant(query))
    print()

## Colab interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display

query_box = widgets.Textarea(
    placeholder='Enter one question per line',
    description='Queries:',
    layout=widgets.Layout(width='90%', height='110px')
)
submit_button = widgets.Button(description='Get responses')
response_area = widgets.Output()

def respond_to_queries(button):
    with response_area:
        response_area.clear_output()
        for query in query_box.value.splitlines():
            if query.strip():
                print('You:', query)
                print('Assistant:', assistant(query))
                print()

submit_button.on_click(respond_to_queries)
display(query_box, submit_button, response_area)

The assistant provides general information only. It does not diagnose conditions or replace advice from a healthcare professional.